# Kaggle: Teacher-generate IRAC SFT targets

Runs `src/data/generate_teacher_targets.py` against the full CUAD generation queue, using **Ollama running on the Kaggle T4** as the teacher backend (not raw HF `transformers`).

**Why Ollama instead of `transformers` here:** an earlier version of this notebook batched generation directly with `transformers`, and needed four separate bug fixes before it worked correctly (right-padding breaking the KV cache, `device_map="auto"` splitting the model pipeline-parallel across a T4x2 session, no bound on a batch's worst-case latency, and bf16 not fitting a single T4 without 4-bit). Ollama (built on llama.cpp) handles padding, batching, and quantization internally — running it here avoids re-solving problems that already have a mature, correct implementation. See `LOG.md` 2026-08-16/17 for the full debugging trail.

**Before running:**
1. Zip the repo's `src/` folder (this notebook imports `src.data.build_irac` and `src.data.generate_teacher_targets` directly — same code path as the local Ollama smoke test, no duplicated logic).
2. Upload that zip, plus `data/raw/sft_generation_queue.jsonl`, as a Kaggle Dataset.
3. Attach the dataset to this notebook (Add Data), turn on a T4 GPU accelerator, and **turn on Internet** (Settings — needed to install/pull Ollama and its model).
4. Run all cells. The module-finder cell searches `/kaggle/input` for the code rather than requiring a hardcoded path (Kaggle's mount path nesting has proven unpredictable across uploads on this account).

**Output:** `sft_teacher_targets.jsonl` written to `/kaggle/working/` — download it and place it at `data/raw/sft_teacher_targets.jsonl` in the local repo to continue the pipeline (corruption functions, splits, rubric).

Resumable: if the session hits Kaggle's runtime limit partway through, re-run with the partially-written output file re-uploaded as input and it will skip already-completed ids.

In [ ]:
!apt-get update -qq && apt-get install -y -qq zstd
!curl -fsSL https://ollama.com/install.sh | sh
!pip install -q requests

In [ ]:
import os, subprocess, time, requests

# Let the Ollama scheduler batch concurrent requests on the GPU (this is
# what replaces the manual padding/batching code from the transformers
# path). Match this to the --concurrency value passed to run() below.
OLLAMA_NUM_PARALLEL = 8
env = os.environ.copy()
env["OLLAMA_NUM_PARALLEL"] = str(OLLAMA_NUM_PARALLEL)

subprocess.Popen(["ollama", "serve"], env=env,
                  stdout=open("/kaggle/working/ollama_serve.log", "w"),
                  stderr=subprocess.STDOUT)

for _ in range(30):
    try:
        requests.get("http://localhost:11434/api/version", timeout=2)
        break
    # Catches both "port not open yet" (ConnectionError) and "port open
    # but server not yet answering requests" (ReadTimeout/Timeout) -- a
    # real run on a sibling notebook hit the latter and crashed since this
    # loop only caught ConnectionError originally. See LOG.md 2026-08-21.
    except (requests.exceptions.ConnectionError, requests.exceptions.Timeout):
        time.sleep(1)
else:
    raise RuntimeError("Ollama server did not come up — check /kaggle/working/ollama_serve.log")
print("Ollama server is up.")

MODEL = "qwen2.5:7b-instruct-q4_0"
subprocess.run(["ollama", "pull", MODEL], check=True)

# `ollama pull` only downloads the model -- it isn't loaded into (GPU) memory
# until the first request. Send a tiny warmup generate so `ollama ps` below
# actually reports something, and so we catch a CPU-fallback here rather
# than 3 hours into the real run.
requests.post("http://localhost:11434/api/generate",
               json={"model": MODEL, "prompt": "hello", "stream": False})
ps = subprocess.run(["ollama", "ps"], capture_output=True, text=True)
print(ps.stdout)
if "100% GPU" not in ps.stdout and "GPU" not in ps.stdout.split("\n")[-2]:
    print("WARNING: model may not be fully on GPU -- check the PROCESSOR column above.")

In [ ]:
import os, sys, zipfile

def find_teacher_gen_module(root="/kaggle/input"):
    """Search recursively for the code, rather than hardcoding a mount path.
    Kaggle's /kaggle/input nesting has proven unpredictable across dataset
    versions/uploads on this account (three different shapes seen so far:
    a flat placeholder guess, /datasets/<user>/<slug>/, and a slug mismatch
    after re-upload) — searching directly removes that whole class of bug."""
    for r, dirs, files in os.walk(root):
        if "generate_teacher_targets.py" in files and os.path.basename(r) == "data":
            src_dir = os.path.dirname(r)
            if os.path.basename(src_dir) == "src":
                return os.path.dirname(src_dir), None  # parent-of-src
        if "src.zip" in files:
            return None, os.path.join(r, "src.zip")
    return None, None

input_dir, zip_path = find_teacher_gen_module()

if zip_path:
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall("/kaggle/working/repo")
    sys.path.insert(0, "/kaggle/working/repo")
    INPUT_DIR = os.path.dirname(zip_path)
    print("Extracted src.zip from", zip_path)
elif input_dir:
    sys.path.insert(0, input_dir)
    INPUT_DIR = input_dir
    print("Found src/ package at", input_dir)
else:
    raise FileNotFoundError(
        "Could not find src/data/generate_teacher_targets.py or src.zip anywhere "
        "under /kaggle/input. Check the dataset is attached in the Input panel "
        "and contains either an unzipped src/ folder or a src.zip. If you just "
        "attached/updated it, try Restart & Run All — Kaggle sometimes needs a "
        "fresh session to mount a newly added input."
    )

from src.data.generate_teacher_targets import run

In [ ]:
import shutil

QUEUE_PATH = os.path.join(INPUT_DIR, "sft_generation_queue.jsonl")
OUTPUT_PATH = "/kaggle/working/sft_teacher_targets.jsonl"

# If resuming a previous partial run, copy its output here first:
prev_output = os.path.join(INPUT_DIR, "sft_teacher_targets.jsonl")
if os.path.exists(prev_output) and not os.path.exists(OUTPUT_PATH):
    shutil.copy(prev_output, OUTPUT_PATH)
    print("Resumed from previous partial output.")

run(
    queue_path=QUEUE_PATH,
    output_path=OUTPUT_PATH,
    model=MODEL,
    host="http://localhost:11434",
    max_rows=None,
    concurrency=OLLAMA_NUM_PARALLEL,
)

In [ ]:
# Quick sanity check before downloading
import json
with open(OUTPUT_PATH, encoding="utf-8") as f:
    lines = [json.loads(l) for l in f]
n_ok = sum(1 for r in lines if r["parse_ok"])
print(f"{len(lines)} rows generated, {n_ok} parsed cleanly ({100*n_ok/len(lines):.1f}%)")
print(lines[0]["target"])